<a href="https://colab.research.google.com/github/Pedro4Albuquerque/Processamento-de-Linguagem-Natural/blob/main/Exerc%C3%ADcio_Pr%C3%A1tico_Aula_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Espaço do aluno — escreva seu código aqui

import re
import unicodedata

corpus_exercicio = [
    "O síndico é muito prestativo e sempre resolve tudo rápido.",
    "A área de lazer não estava limpa quando visitei.",
    "O condomínio é caro, mas a estrutura compensa.",
    "Nunca vi um prédio tão mal administrado.",
]


# ---------------------------------------------------------
# Léxico inicial
# ---------------------------------------------------------

lexico = {
    # Palavras positivas
    "prestativo": 1,
    "resolve": 1,
    "rapido": 1,
    "limpa": 1,
    "bom": 1,
    "otimo": 1,

    # Palavras negativas
    "caro": -1,
    "mal": -1,
    "sujo": -1,
    "ruim": -1,
}


# Palavras que indicam negação
negacoes = {
    "nao",
    "nunca",
    "nem",
    "jamais"
}


# Função para remover acentos e deixar o texto padronizado
def normalizar_texto(texto):
    texto = texto.lower()

    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caractere
        for caractere in texto
        if unicodedata.category(caractere) != "Mn"
    )

    return texto


# Função de polaridade com tratamento simples de negação
def polaridade_com_negacao(texto, lexico):
    texto = normalizar_texto(texto)

    palavras = re.findall(r"\b\w+\b", texto)

    pontuacao = 0
    detalhes = []

    for i, palavra in enumerate(palavras):

        if palavra in lexico:
            valor = lexico[palavra]

            # Verifica até 3 palavras anteriores procurando uma negação
            inicio = max(0, i - 3)
            palavras_anteriores = palavras[inicio:i]

            tem_negacao = any(
                palavra_anterior in negacoes
                for palavra_anterior in palavras_anteriores
            )

            if tem_negacao:
                valor = -valor

            pontuacao += valor

            detalhes.append({
                "palavra": palavra,
                "negacao": tem_negacao,
                "valor": valor
            })

    if pontuacao > 0:
        classificacao = "positivo"
    elif pontuacao < 0:
        classificacao = "negativo"
    else:
        classificacao = "neutro"

    return classificacao, pontuacao, detalhes


# =========================================================
# 1. Classificação de cada comentário
# =========================================================

print("1. CLASSIFICAÇÃO INICIAL\n")

resultados = []

for comentario in corpus_exercicio:

    classificacao, pontuacao, detalhes = polaridade_com_negacao(
        comentario,
        lexico
    )

    resultados.append((comentario, classificacao, pontuacao, detalhes))

    print("Comentário:", comentario)
    print("Classificação:", classificacao)
    print("Pontuação:", pontuacao)
    print("Palavras encontradas:", detalhes)
    print("-" * 70)


# =========================================================
# 2. Investigação dos resultados inesperados
# =========================================================

print("\n2. INVESTIGAÇÃO DOS RESULTADOS\n")

print("""
Comentário 1:
"O síndico é muito prestativo e sempre resolve tudo rápido."

O resultado esperado é positivo.
As palavras "prestativo", "resolve" e "rápido" estão no léxico
e possuem valor positivo. Portanto, o resultado está de acordo
com a leitura humana.
""")


print("""
Comentário 2:
"A área de lazer não estava limpa quando visitei."

A palavra "limpa" seria positiva sozinha, mas existe a palavra
"não" antes dela. A função identifica a negação e inverte
a polaridade de "limpa", fazendo o comentário ficar negativo.

O resultado está de acordo com a leitura humana.
""")


print("""
Comentário 3:
"O condomínio é caro, mas a estrutura compensa."

Neste comentário existe um problema de cobertura do léxico.

A palavra "caro" está cadastrada como negativa, porém
"compensa", que possui sentido positivo nesse contexto,
não existe no léxico inicial.

Por causa disso, inicialmente o algoritmo considera apenas
"caro" e classifica o comentário como negativo.

Na leitura humana, entretanto, a frase apresenta uma ideia
mista: o condomínio é caro, mas sua estrutura compensa.
Por isso, "compensa" deveria participar da análise.
""")


print("""
Comentário 4:
"Nunca vi um prédio tão mal administrado."

O resultado obtido foi negativo, o que está de acordo com
a leitura humana da frase.

A palavra "mal" possui valor negativo no léxico. Embora exista
a palavra "nunca" na frase, ela está distante de "mal" e fica
fora da janela de três palavras usada pela função para detectar
negações.

Nesse caso, portanto, não foi necessário fazer ajuste no léxico,
pois a classificação obtida já representa corretamente o
sentimento negativo do comentário.
""")


# =========================================================
# 3. Ajuste do léxico
# =========================================================

print("\n3. AJUSTE DO LÉXICO\n")

comentario_teste = corpus_exercicio[2]

antes = polaridade_com_negacao(
    comentario_teste,
    lexico
)

print("ANTES DO AJUSTE")
print("Comentário:", comentario_teste)
print("Classificação:", antes[0])
print("Pontuação:", antes[1])
print("Detalhes:", antes[2])


# Adicionando a palavra que estava faltando
lexico["compensa"] = 1


depois = polaridade_com_negacao(
    comentario_teste,
    lexico
)

print("\nDEPOIS DO AJUSTE")
print("Comentário:", comentario_teste)
print("Classificação:", depois[0])
print("Pontuação:", depois[1])
print("Detalhes:", depois[2])


# ---------------------------------------------------------
# Nova classificação de todo o corpus após o ajuste
# ---------------------------------------------------------

print("\nRESULTADO FINAL APÓS O AJUSTE DO LÉXICO\n")

for comentario in corpus_exercicio:

    classificacao, pontuacao, detalhes = polaridade_com_negacao(
        comentario,
        lexico
    )

    print("Comentário:", comentario)
    print("Classificação:", classificacao)
    print("Pontuação:", pontuacao)
    print("-" * 70)

1. CLASSIFICAÇÃO INICIAL

Comentário: O síndico é muito prestativo e sempre resolve tudo rápido.
Classificação: positivo
Pontuação: 3
Palavras encontradas: [{'palavra': 'prestativo', 'negacao': False, 'valor': 1}, {'palavra': 'resolve', 'negacao': False, 'valor': 1}, {'palavra': 'rapido', 'negacao': False, 'valor': 1}]
----------------------------------------------------------------------
Comentário: A área de lazer não estava limpa quando visitei.
Classificação: negativo
Pontuação: -1
Palavras encontradas: [{'palavra': 'limpa', 'negacao': True, 'valor': -1}]
----------------------------------------------------------------------
Comentário: O condomínio é caro, mas a estrutura compensa.
Classificação: negativo
Pontuação: -1
Palavras encontradas: [{'palavra': 'caro', 'negacao': False, 'valor': -1}]
----------------------------------------------------------------------
Comentário: Nunca vi um prédio tão mal administrado.
Classificação: negativo
Pontuação: -1
Palavras encontradas: [{'pala